In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

log_path = "rollout_log.csv"
df = pd.read_csv(log_path)

# Parse stringified arrays
for col in ["observation", "clipped_action", "physical_action"]:
    df[col] = df[col].apply(ast.literal_eval)

# Optional: focus on a single environment rollout
# df = df[df["env_index"] == 0].reset_index(drop=True)

qpos = np.stack(df["observation"].apply(lambda obs: obs[:8]).to_numpy())
clipped = np.stack(df["clipped_action"].apply(lambda act: act[:8]).to_numpy())
physical = np.stack(df["physical_action"].apply(lambda act: act[:8]).to_numpy())
steps = df["step"].to_numpy()

plt.style.use("seaborn-v0_8-darkgrid")

series_style = {
    "qpos": {"color": "#1f77b4", "linewidth": 1.4, "alpha": 1.0, "label": "qpos (blue)"},
    "clipped": {"color": "#ff7f0e", "linewidth": 1.2, "alpha": 0.85, "label": "clipped action (orange)"},
    "physical": {"color": "#2ca02c", "linewidth": 1.2, "alpha": 0.85, "label": "physical action (green)"},
}

fig, axes = plt.subplots(4, 2, figsize=(18, 12), sharex=True)
joint_labels = [f"Joint {idx}" for idx in range(8)]

for idx, ax in enumerate(axes.flat):
    ax.plot(steps, qpos[:, idx], **series_style["qpos"])
    ax.plot(steps, clipped[:, idx], **series_style["clipped"])
    ax.plot(steps, physical[:, idx], **series_style["physical"])
    ax.set_title(joint_labels[idx])
    ax.set_ylabel("Value")

axes[-1, 0].set_xlabel("Step")
axes[-1, 1].set_xlabel("Step")
handles = [
    plt.Line2D([], [], **series_style["qpos"]),
    plt.Line2D([], [], **series_style["clipped"]),
    plt.Line2D([], [], **series_style["physical"]),
]
fig.legend(handles, [h.get_label() for h in handles], loc="upper right")
fig.suptitle("First Eight qpos vs Clipped/Physical Actions", fontsize=18, y=1.02)
fig.tight_layout()
plt.show()

# Scatter plots comparing qpos with clipped and physical actions
fig, axes = plt.subplots(4, 2, figsize=(18, 12))
for idx, ax in enumerate(axes.flat):
    ax.scatter(qpos[:, idx], clipped[:, idx], s=16, alpha=0.45, color="#ff7f0e", label="clipped action")
    ax.scatter(qpos[:, idx], physical[:, idx], s=16, alpha=0.45, color="#2ca02c", marker="x", label="physical action")
    ax.set_title(f"Joint {idx}")
    ax.set_xlabel("qpos")
    ax.set_ylabel("action value")
    ax.axhline(0.0, color="grey", linewidth=0.8)
    ax.axvline(0.0, color="grey", linewidth=0.8)
    ax.legend(loc="upper left")

fig.suptitle("qpos vs Action Scatter", fontsize=18, y=1.02)
fig.tight_layout()
plt.show()

corr = pd.DataFrame({
    "joint": [f"joint_{idx}" for idx in range(8)],
    "pearson_corr_clipped": [np.corrcoef(qpos[:, idx], clipped[:, idx])[0, 1] for idx in range(8)],
    "pearson_corr_physical": [np.corrcoef(qpos[:, idx], physical[:, idx])[0, 1] for idx in range(8)],
})
display(corr)
